[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/24-quantization.ipynb)

# Module 24: Quantization and Number Formats

Companion notebook to [Module 24](https://transformers-and-math.drmhse.com/modules/24-quantization) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — Range and precision of float formats (section 24.1)

In [ ]:
for dtype in (torch.float32, torch.float16, torch.bfloat16):
    info = torch.finfo(dtype)
    print(f'{str(dtype):15} max {info.max:10.3g}   gap near 1 {info.eps:.3g}')
print('1 + 0.001 in bf16 =', (torch.tensor(1.0, dtype=torch.bfloat16) + 0.001).item())
print('70000 in fp16 =', torch.tensor(70000.0).half().item())

## Part 2 — Integer quantization and scale granularity (section 24.2)

In [ ]:
def quantize(w, bits, dim=None):
    qmax = 2 ** (bits - 1) - 1
    scale = w.abs().amax() / qmax if dim is None else w.abs().amax(dim=dim, keepdim=True) / qmax
    return torch.clamp(torch.round(w / scale), -qmax - 1, qmax) * scale

col_scale = torch.exp(torch.randn(64) * 0.8)                # uneven channels, as in real layers
W = torch.randn(256, 64) * 0.05 * col_scale
X = torch.randn(32, 256)
Y = X @ W
for bits in (8, 4, 3):
    for name, dim in (('per tensor', None), ('per channel', 0)):
        err = torch.linalg.norm(Y - X @ quantize(W, bits, dim)) / torch.linalg.norm(Y)
        print(f'INT{bits} {name:11}: relative output error {err:.2%}')

## Part 3 — Outlier channels and SmoothQuant (section 24.3)

In [ ]:
Xo = torch.randn(32, 256); Xo[:, [3, 77]] *= 40           # two outlier activation channels
Wo = torch.randn(256, 64) * 0.05
Yo = Xo @ Wo
def w8a8(X, W):
    return quantize(X, 8) @ quantize(W, 8, 0)
s = (Xo.abs().amax(0) ** 0.5) / (Wo.abs().amax(1) ** 0.5)    # α = 0.5
print('W8A8 error, plain:      ', f'{torch.linalg.norm(Yo - w8a8(Xo, Wo)) / torch.linalg.norm(Yo):.2%}')
print('W8A8 error, SmoothQuant:', f'{torch.linalg.norm(Yo - w8a8(Xo / s, Wo * s[:, None])) / torch.linalg.norm(Yo):.2%}')
print('the scaled product is unchanged:', torch.allclose((Xo / s) @ (Wo * s[:, None]), Yo, atol=1e-3))

## Part 4 — Quantization-aware training with the straight-through estimator (section 24.5)

In [ ]:
class FakeQuant(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, bits):
        return quantize(w, bits, 1)                           # one scale per output row
    @staticmethod
    def backward(ctx, g):
        return g, None                                       # straight-through: pretend rounding is the identity

def run(student, x, bits=None):
    h = x
    for m in student:
        if isinstance(m, nn.Linear):
            w = m.weight if bits is None else FakeQuant.apply(m.weight, bits)
            h = F.linear(h, w, m.bias)
        else:
            h = m(h)
    return h

def train(qat, bits, steps=2000):
    torch.manual_seed(0)
    teacher = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 8))
    student = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 8))
    opt = torch.optim.Adam(student.parameters(), lr=3e-3)
    for _ in range(steps):
        x = torch.randn(256, 32)
        loss = F.mse_loss(run(student, x, bits if qat else None), teacher(x).detach())
        opt.zero_grad(); loss.backward(); opt.step()
    x = torch.randn(4096, 32)
    with torch.no_grad():
        return F.mse_loss(run(student, x, bits), teacher(x)).item()   # always evaluated with quantized weights

for bits in (2, 3):
    print(f'{bits}-bit weights:  quantized after training {train(False, bits):.4f}   quantization-aware training {train(True, bits):.4f}')

At 2 bits, plain rounding destroys much of what the network learned, and training *through* the rounding recovers a good part of it. At 3 bits, rounding after training is already close, and here QAT's straight-through gradients (which ignore the rounding) gain nothing. This is the pattern section 24.5 describes: QAT pays off at the lowest bit widths.

## Your turn
1. Repeat Part 2 with per-group scales (groups of 32 rows per column). How much does INT4 improve?
2. In Part 3, try α = 0.8 and α = 0.2. Which side does each push the difficulty to?